# Week 3 Day 5 — Svensson Validation

Compares our fitted zero rates against the Fed's published GSW rates.

**Key insight:** we compare *curves*, not *parameters*. Svensson is ill-conditioned —
many (β, λ) combinations produce identical curves, so parameter differences are
meaningless. What matters is whether `svensson_zero_rate(mat, *our_params)`
matches the Fed's published `svenyXX` rate.

**Target:** |difference| < 2 bp on 95%+ of days at each maturity.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from termstructure.curves.svensson import validate_svensson_daily

## 1. Run validation

Loads `svensson_params.parquet` (our fits) and `treasury_bonds.parquet` (Fed's
`sveny01`–`sveny30`), then evaluates our fitted curve at each maturity.
Fast — no optimization, just function evaluations.

In [ ]:
val = validate_svensson_daily(maturities=[2, 5, 10, 30])
val.head()

## 2. Pass rate by maturity

Primary diagnostic: what fraction of days meet the 2bp target at each maturity?

If 10Y misses 95%, it usually means optimizer failures on specific dates —
check those dates' RMSE in `svensson_params.parquet`.

In [ ]:
rows = []
for mat in [2, 5, 10, 30]:
    sub = val[val['maturity'] == mat]['diff_bps'].abs()
    rows.append({
        'maturity':       f'{mat}Y',
        'n_days':         len(sub),
        'median_bp':      round(sub.median(), 2),
        '95th_bp':        round(sub.quantile(0.95), 2),
        'pct_within_2bp': f"{(sub < 2.0).mean()*100:.1f}%",
        'pct_within_5bp': f"{(sub < 5.0).mean()*100:.1f}%",
    })
pd.DataFrame(rows).set_index('maturity')

## 3. 10Y comparison: our rate vs. Fed's

Top panel: the two series should be nearly indistinguishable.
Bottom panel: daily difference in basis points — most mass should be inside ±2bp.

In [ ]:
t10 = val[val['maturity'] == 10].set_index('date').sort_index()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 7), sharex=True,
                                gridspec_kw={'height_ratios': [3, 1]})

ax1.plot(t10.index, t10['fed_rate'] * 100, lw=0.8, label="Fed's 10Y (sveny10)", color='tomato')
ax1.plot(t10.index, t10['our_rate'] * 100, lw=0.8, label='Our Svensson fit',
         color='steelblue', alpha=0.8)
ax1.set_ylabel('Zero yield (%)')
ax1.set_title('10Y zero rate: our Svensson fit vs. Fed\'s published rate')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.fill_between(t10.index, t10['diff_bps'], 0,
                 where=t10['diff_bps'] >= 0, color='steelblue', alpha=0.5)
ax2.fill_between(t10.index, t10['diff_bps'], 0,
                 where=t10['diff_bps'] < 0,  color='tomato',    alpha=0.5)
ax2.axhline(0,   color='black', lw=0.8)
ax2.axhline(2,   color='gray',  lw=0.8, ls='--', label='±2bp target')
ax2.axhline(-2,  color='gray',  lw=0.8, ls='--')
ax2.set_ylabel('Diff (bp)')
ax2.legend(loc='upper right', fontsize=8)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('data/week3_day5_validation.png', dpi=150)
plt.show()

within = (t10['diff_bps'].abs() < 2.0).mean()
print(f'10Y: {within*100:.1f}% of days within 2bp  (target: 95%)')

## 4. Error distribution at 10Y

Most mass should be inside ±2bp. A heavy tail toward large positive or negative
errors points to specific dates where the optimizer found a bad local minimum —
those dates will also have high RMSE in `svensson_params.parquet`.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(t10['diff_bps'], bins=100, color='steelblue', alpha=0.7, edgecolor='none')
ax.axvline( 2, color='tomato', lw=1.5, ls='--', label='±2bp')
ax.axvline(-2, color='tomato', lw=1.5, ls='--')
ax.set_xlabel('Difference (bp)  [ours − Fed]')
ax.set_ylabel('Days')
ax.set_title('Distribution of 10Y fit error')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Worst-fit dates

The dates with the largest 10Y error. These are optimizer failures —
you can cross-reference them with the RMSE column in `svensson_params.parquet`.

In [ ]:
worst = t10['diff_bps'].abs().nlargest(10)
print('10 worst days (|diff| in bp):')
print(t10.loc[worst.index, ['our_rate', 'fed_rate', 'diff_bps']]
      .assign(our_rate=lambda d: (d.our_rate * 100).round(4),
              fed_rate=lambda d: (d.fed_rate * 100).round(4),
              diff_bps=lambda d: d.diff_bps.round(2)))